# Mag7 Sma Crossover Comparison

Interactive notebook version of this workflow. Edit the parameters and rerun cells to experiment.


In [ ]:
from __future__ import annotations

from time import perf_counter

import pandas as pd

from quant_orchestrator.platforms.backtesting_frameworks.backtesting_py.data_adapter import (
    build_backtesting_frame,
)
from quant_orchestrator.platforms.backtesting_frameworks.nautilus.data_adapter import (
    build_nautilus_in_memory_data,
)
from quant_orchestrator.platforms.backtesting_frameworks.shared import (
    MAG7_SYMBOLS,
    load_signal_frame,
    normalize_session_label,
)
from quant_orchestrator.platforms.backtesting_frameworks.zipline.data_adapter import (
    build_zipline_in_memory_data,
)
from quant_orchestrator.strategy import summarize_backtest


SYMBOL_COUNT = len(MAG7_SYMBOLS)


def _patch_zipline_compatibility() -> None:
    if getattr(_patch_zipline_compatibility, "_patched", False):
        return

    import zipline.finance.ledger as ledger_mod
    from zipline.data.in_memory_daily_bars import InMemoryDailyBarReader

    InMemoryDailyBarReader.frames = property(lambda self: self._frames)
    ledger_mod.PositionTracker.stats = property(lambda self: self._stats)
    _patch_zipline_compatibility._patched = True  # type: ignore[attr-defined]


def run_backtesting_py(frame: pd.DataFrame, *, symbol: str, capital_base: float):
    from backtesting import Backtest, Strategy

    trade_size = max(1, int((capital_base * 0.25) / float(frame["close"].iloc[0])))
    bt_frame = build_backtesting_frame(frame).drop(columns=["Signal"])
    signal_map = {
        normalize_session_label(date): bool(signal)
        for date, signal in frame["signal"].items()
    }

    class SignalStrategy(Strategy):
        def init(self):
            self.trade_size = trade_size

        def next(self):
            bullish = signal_map.get(normalize_session_label(self.data.index[-1]), False)
            if bullish and not self.position:
                self.buy(size=self.trade_size)
            elif not bullish and self.position:
                self.position.close()

    started = perf_counter()
    stats = Backtest(
        bt_frame,
        SignalStrategy,
        cash=capital_base,
        commission=0.0,
        trade_on_close=False,
        exclusive_orders=True,
        finalize_trades=True,
    ).run()
    elapsed = perf_counter() - started
    equity = stats["_equity_curve"]["Equity"].rename("portfolio_value")
    summary = summarize_backtest(
        framework="backtesting.py",
        symbol=symbol,
        equity=equity,
        elapsed_seconds=elapsed,
        bars=len(bt_frame),
        trades=int(stats["# Trades"]),
    )
    summary["native_return_pct"] = float(stats["Return [%]"])
    summary["native_sharpe"] = float(stats["Sharpe Ratio"]) if pd.notna(stats["Sharpe Ratio"]) else None
    summary["native_max_drawdown_pct"] = float(stats["Max. Drawdown [%]"])
    return stats, summary, equity


def run_zipline(frame: pd.DataFrame, *, symbol: str, capital_base: float):
    from zipline.algorithm import TradingAlgorithm
    from zipline.api import order_target, record, symbol as zipline_symbol

    _patch_zipline_compatibility()
    adapter = build_zipline_in_memory_data(frame, symbol=symbol, capital_base=capital_base)
    trade_size = adapter.trade_size

    def initialize(context, **kwargs):
        context.asset = zipline_symbol(symbol.upper())
        context.is_long = False

    def handle_data(context, data):
        dt = normalize_session_label(context.get_datetime())
        bullish = adapter.signal_map.get(dt, False)
        if bullish and not context.is_long:
            order_target(context.asset, trade_size)
            context.is_long = True
        elif not bullish and context.is_long:
            order_target(context.asset, 0)
            context.is_long = False
        record(price=data.current(context.asset, "price"), signal=float(bullish))

    started = perf_counter()
    algo = TradingAlgorithm(
        sim_params=adapter.sim_params,
        data_portal=adapter.data_portal,
        asset_finder=adapter.asset_finder,
        initialize=initialize,
        handle_data=handle_data,
        capital_base=capital_base,
        benchmark_returns=adapter.benchmark_returns,
    )
    perf = algo.run()
    elapsed = perf_counter() - started
    transactions = perf.get("transactions", pd.Series(index=perf.index, data=[[]] * len(perf)))
    equity = perf["portfolio_value"].rename("portfolio_value")
    summary = summarize_backtest(
        framework="zipline",
        symbol=symbol,
        equity=equity,
        elapsed_seconds=elapsed,
        bars=len(perf),
        trades=int(transactions.map(len).sum()),
    )
    summary["native_last_value"] = float(perf["portfolio_value"].iloc[-1])
    return perf, summary, equity


def run_nautilus(frame: pd.DataFrame, *, symbol: str, capital_base: float):
    from decimal import Decimal

    from nautilus_trader.backtest.engine import BacktestEngine
    from nautilus_trader.config import BacktestEngineConfig, LoggingConfig, StrategyConfig
    from nautilus_trader.model.currencies import USD
    from nautilus_trader.model.enums import AccountType, OmsType, OrderSide, TimeInForce
    from nautilus_trader.model.objects import Money, Quantity
    from nautilus_trader.trading.strategy import Strategy

    adapter = build_nautilus_in_memory_data(frame, symbol=symbol, capital_base=capital_base)

    class SignalStrategyConfig(StrategyConfig, frozen=True):
        instrument_id: object
        bar_type: object
        trade_size: Decimal
        signal_map: dict

    class SignalStrategy(Strategy):
        def __init__(self, config: SignalStrategyConfig):
            super().__init__(config)
            self.is_long = False

        def on_start(self) -> None:
            self.subscribe_bars(self.config.bar_type)

        def on_bar(self, bar) -> None:
            bullish = self.config.signal_map.get(normalize_session_label(bar.ts_event), False)
            if bullish == self.is_long:
                return
            side = OrderSide.BUY if bullish else OrderSide.SELL
            order = self.order_factory.market(
                instrument_id=self.config.instrument_id,
                order_side=side,
                quantity=Quantity.from_int(int(self.config.trade_size)),
                time_in_force=TimeInForce.IOC,
            )
            self.submit_order(order)
            self.is_long = bullish

    engine = BacktestEngine(
        config=BacktestEngineConfig(
            logging=LoggingConfig(log_level="OFF", bypass_logging=True),
        ),
    )
    engine.add_venue(
        venue=adapter.venue,
        oms_type=OmsType.NETTING,
        account_type=AccountType.CASH,
        starting_balances=[Money(capital_base, USD)],
        base_currency=USD,
    )
    engine.add_instrument(adapter.instrument)
    engine.add_data(adapter.bars)
    engine.add_strategy(
        SignalStrategy(
            SignalStrategyConfig(
                instrument_id=adapter.instrument.id,
                bar_type=adapter.bar_type,
                trade_size=Decimal(adapter.trade_size),
                signal_map=adapter.signal_map,
            ),
        ),
    )

    started = perf_counter()
    engine.run()
    fills_report = engine.trader.generate_order_fills_report()
    elapsed = perf_counter() - started
    engine.dispose()

    equity = _equity_from_fills(prices=frame, fills=fills_report, capital_base=capital_base)
    summary = summarize_backtest(
        framework="nautilus",
        symbol=symbol,
        equity=equity,
        elapsed_seconds=elapsed,
        bars=len(frame),
        trades=len(fills_report),
    )
    summary["native_fills"] = int(len(fills_report))
    summary["native_last_value"] = float(equity.iloc[-1])
    return fills_report, summary, equity


def _equity_from_fills(*, prices: pd.DataFrame, fills: pd.DataFrame, capital_base: float) -> pd.Series:
    cash = float(capital_base)
    position = 0.0
    values = []
    fills_by_date: dict[pd.Timestamp, list[pd.Series]] = {}

    for _, fill in fills.iterrows():
        fill_date = normalize_session_label(fill["ts_last"])
        fills_by_date.setdefault(fill_date, []).append(fill)

    for date, row in prices.iterrows():
        normalized = normalize_session_label(date)
        for fill in fills_by_date.get(normalized, []):
            quantity = float(fill["filled_qty"])
            price = float(fill["avg_px"])
            if str(fill["side"]) == "BUY":
                cash -= quantity * price
                position += quantity
            else:
                cash += quantity * price
                position -= quantity
        values.append(cash + position * float(row["close"]))

    return pd.Series(values, index=prices.index, name="portfolio_value")


def _print_native_snapshot(framework: str, raw_result) -> None:
    if framework == "backtesting.py":
        cols = [
            "Start",
            "End",
            "Equity Final [$]",
            "Return [%]",
            "Sharpe Ratio",
            "Max. Drawdown [%]",
            "# Trades",
        ]
        print(raw_result.loc[cols])
        return

    if framework == "zipline":
        print(raw_result.loc[:, ["portfolio_value", "returns"]].tail())
        return

    if framework == "nautilus":
        print(raw_result.loc[:, ["side", "filled_qty", "avg_px", "ts_last"]].head())
        return

    print(raw_result)


def run_symbol_comparison(symbol: str, *, provider: str = "yfinance", start: str | None = "2020-01-01", end: str | None = None, capital_base: float = 100_000.0):
    frame = load_signal_frame(symbol, provider=provider, start=start, end=end)
    raw_results = {
        "backtesting.py": run_backtesting_py(frame, symbol=symbol, capital_base=capital_base),
        "zipline": run_zipline(frame, symbol=symbol, capital_base=capital_base),
        "nautilus": run_nautilus(frame, symbol=symbol, capital_base=capital_base),
    }
    summary = pd.concat([result[1] for result in raw_results.values()], ignore_index=True)
    return summary, raw_results


def run_mag7_comparison(*, provider: str = "yfinance", start: str | None = "2020-01-01", end: str | None = None, capital_base: float = 100_000.0):
    summaries = []
    runs: dict[str, dict[str, object]] = {}
    framework_equity: dict[str, list[pd.Series]] = {}
    capital_per_symbol = capital_base / SYMBOL_COUNT
    for symbol in MAG7_SYMBOLS:
        summary, raw_results = run_symbol_comparison(
            symbol,
            provider=provider,
            start=start,
            end=end,
            capital_base=capital_per_symbol,
        )
        summaries.append(summary)
        runs[symbol.upper()] = raw_results

        for framework, (_, _, equity) in raw_results.items():
            framework_equity.setdefault(framework, []).append(equity.rename(symbol.upper()))

    summary = pd.concat(summaries, ignore_index=True)
    portfolio_rows = []
    for framework, sleeve_equities in framework_equity.items():
        combined_index = pd.Index([])
        for equity in sleeve_equities:
            combined_index = combined_index.union(equity.index)
        combined = pd.Series(0.0, index=combined_index)
        for equity in sleeve_equities:
            reindexed = equity.reindex(combined_index)
            reindexed = reindexed.ffill()
            reindexed = reindexed.fillna(equity.iloc[0])
            combined = combined.add(reindexed, fill_value=0.0)
        portfolio_rows.append(
            summarize_backtest(
                framework=framework,
                symbol="MAG7",
                equity=combined.sort_index(),
                elapsed_seconds=float(summary.loc[summary["framework"] == framework, "elapsed_seconds"].sum()),
                bars=len(combined),
                trades=int(summary.loc[summary["framework"] == framework, "trades"].sum()),
            )
        )
    portfolio_summary = pd.concat(portfolio_rows, ignore_index=True)
    return summary, portfolio_summary, runs


def main() -> None:
    summary, portfolio_summary, runs = run_mag7_comparison(start="2020-01-01")

    print("MAG7 symbols:")
    print(", ".join(MAG7_SYMBOLS))
    print(f"Equal notional per symbol: 1/{SYMBOL_COUNT} of capital_base")
    print()
    print("Summary by symbol and framework:")
    print(summary.to_string(index=False))
    print()
    print("Portfolio summary by framework:")
    print(portfolio_summary.to_string(index=False))
    print()
    print("Summary by framework:")
    grouped = portfolio_summary.groupby("framework", as_index=False).agg(
        mean_total_return=("total_return", "mean"),
        mean_max_drawdown=("max_drawdown", "mean"),
        mean_elapsed_seconds=("elapsed_seconds", "mean"),
        total_elapsed_seconds=("elapsed_seconds", "sum"),
        total_trades=("trades", "sum"),
    )
    print(grouped.to_string(index=False))
    print()

    for symbol in MAG7_SYMBOLS:
        print(f"=== {symbol} ===")
        for framework, (raw_result, _, _) in runs[symbol].items():
            print(f"[{framework}]")
            _print_native_snapshot(framework, raw_result)
            print()


if __name__ == "__main__":
    main()
